# CI/CD for MLflow Models

## Topic Goal


A real CI/CD workflow for ML models should not only train a model. It should also:

1. Check folder and dependency readiness.
2. Validate dataset availability and schema.
3. Train a model.
4. Evaluate the model.
5. Apply a quality gate.
6. Log metrics, artifacts, signature, and input example in the **same MLflow run**.
7. Register the model **only if the quality gate passes**.
8. Generate CI/CD style files such as:
   - `train_pipeline.py`
   - `github_actions_mlflow_pipeline.yml`
   - `deployment_command.txt`
9. Keep an audit trail for automation.

This notebook is designed for classroom demonstration and local execution on Windows/Mac.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, and helper libraries.

This notebook also uses `json` and `textwrap` to generate CI/CD-related files.


In [ ]:
# Import os to create folders and manage environment variables.
import os

# Import sys to check Python version and runtime information.
import sys

# Import json to save configuration-like files.
import json

# Import textwrap to create readable generated scripts/YAML files.
import textwrap

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking, model logging, and registry.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature for capturing model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and processing CSV data.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting dataset.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for different preprocessing per feature type.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for model training.
from sklearn.ensemble import RandomForestClassifier

# Import metrics for model evaluation.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This section prepares local folders similar to what a CI/CD pipeline would prepare before running training.

For this local demo, MLflow uses:

```text
file:./mlruns
```


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "12_cicd_mlflow_models"

# Define run name for the CI/CD workflow.
RUN_NAME = "12_cicd_mlflow_models_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for generated CI/CD artifacts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for support files.
os.makedirs("artifacts", exist_ok=True)

# Configure MLflow to use local file-based tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Define registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print configuration details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)
print("Python version:", sys.version)


## 3. CI/CD Step 1: Environment and Folder Readiness Check

A CI/CD pipeline normally checks whether the required folders and files exist before training starts.

Here we validate:

- dataset path
- output folder
- artifacts folder
- MLflow tracking URI


In [ ]:
# Create a dictionary to store readiness check results.
readiness_checks = {
    # Check whether the dataset file exists.
    "dataset_exists": os.path.exists(DATA_PATH),

    # Check whether outputs folder exists.
    "outputs_folder_exists": os.path.exists("outputs"),

    # Check whether artifacts folder exists.
    "artifacts_folder_exists": os.path.exists("artifacts"),

    # Check whether mlruns folder exists.
    "mlruns_folder_exists": os.path.exists("mlruns"),

    # Check whether MLflow tracking URI is configured.
    "mlflow_tracking_uri": mlflow.get_tracking_uri(),
}

# Display readiness checks.
readiness_checks


In [ ]:
# Stop execution early if dataset is missing.
if not readiness_checks["dataset_exists"]:
    # Raise a clear error for students.
    raise FileNotFoundError(f"Dataset not found at path: {DATA_PATH}")

# Print success message.
print("Environment readiness check passed.")


## 4. CI/CD Step 2: Dataset Validation

Before training, a CI/CD pipeline should validate the dataset schema.

We check:

- required columns
- missing columns
- row count
- target column availability


In [ ]:
# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define required columns.
required_columns = [
    "customer_id",
    "age",
    "gender",
    "region",
    "plan_type",
    "tenure_months",
    "monthly_charges",
    "support_tickets",
    "data_usage_gb",
    "contract_type",
    "churn",
]

# Find missing columns.
missing_columns = [column for column in required_columns if column not in df.columns]

# Check whether dataset schema is valid.
schema_valid = len(missing_columns) == 0

# Check row count.
row_count = len(df)

# Define minimum expected rows.
minimum_rows = 100

# Check whether row count is acceptable.
row_count_valid = row_count >= minimum_rows

# Create dataset validation result.
dataset_validation = {
    "schema_valid": schema_valid,
    "missing_columns": missing_columns,
    "row_count": row_count,
    "minimum_rows": minimum_rows,
    "row_count_valid": row_count_valid,
}

# Display dataset validation result.
dataset_validation


In [ ]:
# Stop execution if schema validation fails.
if not schema_valid:
    # Raise clear error showing missing columns.
    raise ValueError(f"Dataset schema validation failed. Missing columns: {missing_columns}")

# Stop execution if row count is too low.
if not row_count_valid:
    # Raise clear error showing row count issue.
    raise ValueError(f"Dataset row count validation failed. Rows found: {row_count}")

# Print success message.
print("Dataset validation passed.")


## 5. CI/CD Step 3: Prepare Training Data

After validation, we prepare features, target, preprocessing, and train-test split.


In [ ]:
# Define target column.
target_column = "churn"

# Create feature matrix by dropping customer ID and target column.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print feature information.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 6. CI/CD Step 4: Train and Evaluate Model

This represents the training stage in a CI/CD pipeline.

The model is trained and evaluated before deciding whether it can be registered.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input columns.
    ("preprocessor", preprocessor),

    # Train model after preprocessing.
    ("model", model),
])

# Train pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions on test set.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 7. CI/CD Step 5: Quality Gate

A CI/CD workflow should prevent poor models from being registered or deployed.

Here, we define a simple quality gate:

```text
F1-score >= minimum_f1
Recall >= minimum_recall
```


In [ ]:
# Define minimum F1-score threshold.
minimum_f1 = 0.55

# Define minimum recall threshold.
minimum_recall = 0.50

# Check whether model passes quality gate.
passed_quality_gate = (f1 >= minimum_f1) and (recall >= minimum_recall)

# Create quality gate status.
quality_gate_status = "passed" if passed_quality_gate else "failed"

# Print quality gate result.
print("Quality gate status:", quality_gate_status)
print("F1-score:", f1, "| Required:", minimum_f1)
print("Recall:", recall, "| Required:", minimum_recall)


## 8. CI/CD Step 6: Generate CI/CD Artifacts

A CI/CD implementation usually has scripts or YAML files.

This notebook generates:

- `outputs/pipeline_steps.json`
- `outputs/train_pipeline.py`
- `outputs/github_actions_mlflow_pipeline.yml`
- `outputs/deployment_command.txt`
- `outputs/classification_report.txt`

These files are logged as MLflow artifacts.


In [ ]:
# Define CI/CD pipeline steps.
pipeline_steps = [
    "checkout_code",
    "setup_python",
    "install_dependencies",
    "validate_dataset",
    "train_model",
    "evaluate_model",
    "quality_gate",
    "log_model_with_signature",
    "conditional_register_model",
    "prepare_deployment_command",
]

# Save pipeline steps as JSON.
with open("outputs/pipeline_steps.json", "w") as file:
    json.dump(pipeline_steps, file, indent=4)

# Save classification report.
with open("outputs/classification_report.txt", "w") as file:
    file.write(report)

# Create deployment command.
deployment_command = "mlflow models serve -m <MODEL_URI_FROM_CI_RUN>/model -p 5001 --env-manager local"

# Save deployment command.
with open("outputs/deployment_command.txt", "w") as file:
    file.write(deployment_command)

# Create a simple train pipeline script template.
train_pipeline_script = textwrap.dedent('''
    """
    train_pipeline.py

    This is a generated CI/CD training script template.
    In a real project, this file would be committed to source control
    and executed by a CI/CD tool such as GitHub Actions, Jenkins, or Azure DevOps.
    """

    import mlflow

    def validate_data():
        print("Validate dataset schema and row count")

    def train_model():
        print("Train model and calculate metrics")

    def quality_gate():
        print("Check F1-score and recall thresholds")

    def register_model():
        print("Register model only if quality gate passes")

    if __name__ == "__main__":
        validate_data()
        train_model()
        quality_gate()
        register_model()
''').strip()

# Save train pipeline script.
with open("outputs/train_pipeline.py", "w") as file:
    file.write(train_pipeline_script)

# Create GitHub Actions style workflow template.
github_actions_yaml = textwrap.dedent('''
    name: MLflow Model CI Pipeline

    on:
      workflow_dispatch:
      push:
        branches:
          - main

    jobs:
      train-validate-register:
        runs-on: ubuntu-latest

        steps:
          - name: Checkout repository
            uses: actions/checkout@v4

          - name: Set up Python
            uses: actions/setup-python@v5
            with:
              python-version: "3.11"

          - name: Install dependencies
            run: |
              pip install -r requirements.txt

          - name: Run training pipeline
            run: |
              python train_pipeline.py

          - name: Notes
            run: |
              echo "Model registration should happen only if quality gate passes."
''').strip()

# Save GitHub Actions YAML.
with open("outputs/github_actions_mlflow_pipeline.yml", "w") as file:
    file.write(github_actions_yaml)

# Print generated artifact list.
print("Generated CI/CD artifacts:")
print("- outputs/pipeline_steps.json")
print("- outputs/train_pipeline.py")
print("- outputs/github_actions_mlflow_pipeline.yml")
print("- outputs/deployment_command.txt")
print("- outputs/classification_report.txt")


## 9. CI/CD Step 7: Infer Signature

The model signature is required before logging the model.

It documents:

- expected input columns
- expected input data types
- output structure


In [ ]:
# Create input example from test data.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 10. CI/CD Step 8: Same-Run MLflow Logging

This is the main CI/CD tracking run.

Inside the **same MLflow run**, we log:

- data validation result
- quality gate result
- metrics
- artifacts
- model
- input example
- signature
- model URI

Registration happens after this run, but only if the quality gate passes.


In [ ]:
# Start the SAME MLflow run for the CI/CD workflow.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log experiment/topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log dataset validation results.
    mlflow.log_param("schema_valid", schema_valid)
    mlflow.log_param("row_count", row_count)
    mlflow.log_param("row_count_valid", row_count_valid)

    # Log quality gate thresholds.
    mlflow.log_param("minimum_f1", minimum_f1)
    mlflow.log_param("minimum_recall", minimum_recall)

    # Log quality gate result.
    mlflow.log_param("passed_quality_gate", passed_quality_gate)

    # Set quality gate status as tag.
    mlflow.set_tag("quality_gate_status", quality_gate_status)

    # Set CI/CD pipeline stage.
    mlflow.set_tag("pipeline_stage", "train_validate_register")

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Log generated CI/CD artifacts.
    mlflow.log_artifact("outputs/pipeline_steps.json")
    mlflow.log_artifact("outputs/train_pipeline.py")
    mlflow.log_artifact("outputs/github_actions_mlflow_pipeline.yml")
    mlflow.log_artifact("outputs/deployment_command.txt")
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log model with signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from the same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Print same-run model URI.
print("Same-run model URI:", model_uri)


## 11. CI/CD Step 9: Conditional Model Registration

This is the corrected CI/CD behavior.

The model should be registered only if the quality gate passes.

If it fails, the model remains logged for audit, but registration is skipped.


In [ ]:
# Register model only if quality gate passed.
if passed_quality_gate:
    # Register model using the same-run model URI.
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=REGISTERED_MODEL_NAME
    )

    # Print registration success.
    print("Quality gate passed.")
    print("Model registered successfully.")
    print("Registered model name:", registered_model.name)
    print("Registered model version:", registered_model.version)

else:
    # Do not register failed model.
    registered_model = None

    # Print registration skipped message.
    print("Quality gate failed.")
    print("Model was logged for audit but was NOT registered.")
    print("Model URI:", model_uri)
    print("F1-score:", f1, "| Required:", minimum_f1)
    print("Recall:", recall, "| Required:", minimum_recall)


## 12. CI/CD Step 10: Final Pipeline Summary

This section creates a simple summary table for students.


In [ ]:
# Create summary table.
pipeline_summary = pd.DataFrame([
    {"step": "environment_readiness", "status": "passed"},
    {"step": "dataset_schema_validation", "status": "passed" if schema_valid else "failed"},
    {"step": "dataset_row_count_validation", "status": "passed" if row_count_valid else "failed"},
    {"step": "model_training", "status": "completed"},
    {"step": "model_evaluation", "status": "completed"},
    {"step": "quality_gate", "status": quality_gate_status},
    {"step": "model_logging", "status": "completed"},
    {"step": "model_registration", "status": "completed" if registered_model is not None else "skipped"},
])

# Save summary table.
pipeline_summary.to_csv("outputs/cicd_pipeline_summary.csv", index=False)

# Display summary.
display(pipeline_summary)




- CI/CD for ML models is not only code deployment.
- It includes data validation, model training, evaluation, quality gates, registration, and deployment preparation.
- MLflow acts as the traceability layer.
- The same run stores metrics, artifacts, signature, model, and quality-gate status.
- The model is registered only if the quality gate passes.
- Failed models are still useful because they create audit evidence.
- Generated files such as YAML and pipeline scripts show how this notebook maps to real CI/CD systems.
